# Data pipeline example

What do more complex real-word examples of data pipelines look like? How to organize different preprocessing operations?
This module presents a high level example of how audio-visual speech recognition data is processed before it is passed to an audio-visual speech recognition model.

The dataset in this example has the following properties:
  - Each sample is a $224 \times 224$ full-color video of a speakers face with durations ranging between 1-15 seconds
  - In total, the dataset contains 639k samples which corresponds to 1141 hours of video
  - Dataset is divided into pre-train, train, dev and test sets
  - Data is packaged into little over 100 individual shards

The model to be trained expects $88\times88$ grayscale crops of the speaker mouth as image input and log mel-spectrograms as audio input.
Getting the data into the desired form can be split into two stages:
  1) heavy and deterministic single-pass preprocessing of the raw input into intermediary saved format
  2) light-weight, random and transient feature transforms in dataloader


## Stage 1: heavy preprocessing

The video frames in the original raw data in this case are too large for the needs of the model, and need to processed into a more compact form.
Essentially, we would like to apply the following four steps demonstrated by the image below to the video frames of our raw data.

![An example of the steps in heavy preprocessing.](images/preprocess_example.png)

The steps can be summarized as:
  1. face bounding box detection using an existing neural network face detection model
  2. detection of facial landmarks within the bounding box using an existing neural net for the purpose
  3. affine transform of the frame to ensure the face is upright
  4. crop the frame around the mouth region
   
The first two steps involve inference with two different computer vision models, applied to over 100 million individual frames.
That demands large computational resources, but fortunately this process can be considered **deterministic**.
We expect to get the same cropped mouths out each time the process is repeated, thus it makes sense to process all data once and save the output.

Note that this process touches only the video frames, and not the audio, because the audio-side doesn't require any heavy transforms.

## Stage 2: light feature transforms

The grayscale mouth crop videos could already be fed to the model in theory, but there are still some smaller computationally lighter transforms we would like to apply.
And in this case, both input modalities, audio and video, need different treatment.

For video frames, we would like to apply image transforms like random horizontal flip during training, image normalization and tensor datatype transform to ensure our tensors are in the float32 format. For audio features, we want to convert the raw audio signal into log Mel-spectrogram features (a type of audio feature) which is easy and light-weight to compute.

All of these transforms are implemented in the [torchvision](https://docs.pytorch.org/vision/stable/transforms.html#transform-classes-functionals-and-kernels) and [torchaudio](https://docs.pytorch.org/audio/stable/transforms.html) packages under modules `torchvision.transforms` and `torchaudio.transforms`.
These modules are a good place to start if you are wondering how to apply common transformations to your image or audio data.

## Key takeaway

Before implementing your data processing pipelines, consider this: is your data in a form that can be directly fed to the model?
If not, and massaging the data into the required format involves heavy computations, make a separate process that converts your data into the required format.
Only apply light transforms when loading data for the model.